# YOLOv8 3개 부류(cup · bottle · laptop) 학습 — Google Colab

**사용 순서**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 왼쪽 📁 파일 탭에 `cup_bottle_laptop.zip` 업로드 (업로드가 끝날 때까지 기다리기)
3. **런타임 → 모두 실행**
4. 마지막 셀이 `yolo_train_results.zip` 을 다운로드함 → 로컬 프로젝트 폴더(`C:\PROJECT\YOLO`)에 넣기

데이터: COCO 2017 train 에서 부류별 200장 선택 → 총 600장 (train 480 / val 120)

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Ultralytics 설치

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 3. 데이터셋 압축 해제 및 `data.yaml` 경로 수정

In [ ]:
import os, zipfile
from pathlib import Path

ZIP = Path('/content/cup_bottle_laptop.zip')
# Google Drive 에 올렸다면 아래 두 줄의 주석을 풀고 경로를 맞추세요.
# from google.colab import drive; drive.mount('/content/drive')
# ZIP = Path('/content/drive/MyDrive/cup_bottle_laptop.zip')
assert ZIP.exists(), f'{ZIP} 가 없습니다. 왼쪽 파일 탭에 zip 을 업로드하세요.'

DATA_ROOT = Path('/content/datasets')
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DATA_ROOT)
DS = DATA_ROOT / 'cup_bottle_laptop'

yaml_text = (DS / 'data.yaml').read_text().splitlines()
yaml_text[0] = f'path: {DS}'          # 로컬(Windows) 경로 -> Colab 경로
(DS / 'data.yaml').write_text('\n'.join(yaml_text) + '\n')
print((DS / 'data.yaml').read_text())
for s in ('train', 'val'):
    print(s, len(list((DS / 'images' / s).glob('*.jpg'))), 'images')

## 4. 학습 (YOLOv8n, COCO 사전학습 가중치에서 미세조정) — 2차 실험

**1차 실험 결과(기본 설정, epochs=50, patience=15)**: epoch 1 에서 mAP50 0.548 로 최고였고,
warmup 동안 0.26 까지 떨어졌다가 회복 중 16 epoch 에서 조기 종료됨.
→ 원인: warmup 초반의 큰 bias 학습률(`warmup_bias_lr=0.1`)이 사전학습 특징을 망가뜨림 (작은 데이터셋 미세조정)

**2차 실험 변경점**
- `freeze=10` : backbone(0~9층) 고정 → 480장으로도 COCO 특징 유지
- `optimizer='AdamW', lr0=0.001, warmup_bias_lr=0.0, warmup_epochs=1` : 초반 급격한 갱신 방지
- `epochs=100, patience=30, cos_lr=True` : 충분히 학습

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'cup_bottle_laptop_v2'
model = YOLO('yolov8n.pt')
model.train(
    data=str(DS / 'data.yaml'),
    epochs=100,
    patience=30,
    imgsz=640,
    batch=16,
    device=0,
    freeze=10,
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=1,
    warmup_bias_lr=0.0,
    cos_lr=True,
    seed=0,
    project='/content/runs/detect',
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

## 5. 검증 성능 (best.pt)

In [ ]:
import json
RUN = Path('/content/runs/detect') / RUN_NAME
best = YOLO(RUN / 'weights' / 'best.pt')
m = best.val(data=str(DS / 'data.yaml'), split='val', plots=True,
             project='/content/runs/detect', name=RUN_NAME + '_val', exist_ok=True)

summary = {
    'precision': float(m.box.mp), 'recall': float(m.box.mr),
    'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
    'per_class': {best.names[c]: {'precision': float(m.box.p[i]), 'recall': float(m.box.r[i]),
                                  'mAP50': float(m.box.ap50[i]), 'mAP50-95': float(m.box.ap[i])}
                  for i, c in enumerate(m.box.ap_class_index)},
    'speed_ms': m.speed,
}
(RUN / 'final_metrics.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 6. 학습 곡선 · Confusion matrix 확인

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    p = RUN / f
    if p.exists():
        print(f); display(Image(filename=str(p), width=900))

## 7. 결과 다운로드 → 로컬 `C:\PROJECT\YOLO` 폴더에 넣기

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f'/content/yolo_train_results_{RUN_NAME}', 'zip', '/content/runs/detect')
files.download(f'/content/yolo_train_results_{RUN_NAME}.zip')